# Indian Address NER fine-tuning
Fine-tune `xlm-roberta-base` on the generated JSONL dataset. Designed for free Google Colab.

Labels: HOUSE, STREET, LANDMARK, AREA, CITY, STATE, PIN.

In [ ]:
!pip -q install -U transformers datasets evaluate seqeval accelerate huggingface_hub

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload ml/synthetic_addresses.jsonl generated by the script


In [ ]:
import json
from datasets import Dataset

path = next(iter(uploaded))
rows = [json.loads(line) for line in open(path, encoding='utf-8')]
ds = Dataset.from_list(rows).train_test_split(test_size=0.15, seed=42)
ds

In [ ]:
from transformers import AutoTokenizer

MODEL = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
labels = ['O'] + [f'{p}-{x}' for x in ['HOUSE','STREET','LANDMARK','AREA','CITY','STATE','PIN'] for p in ['B','I']]
label2id = {x:i for i,x in enumerate(labels)}
id2label = {i:x for x,i in label2id.items()}

def tokenize_and_align(examples):
    out = tokenizer(examples['tokens'], is_split_into_words=True, truncation=True, max_length=128)
    aligned=[]
    for i, tags in enumerate(examples['ner_tags']):
        word_ids = out.word_ids(batch_index=i)
        previous=None; ids=[]
        for wid in word_ids:
            if wid is None: ids.append(-100)
            elif wid != previous: ids.append(label2id[tags[wid]])
            else:
                # Repeat I-tags for subwords; repeat O as O.
                tag=tags[wid]; ids.append(label2id[tag if tag.startswith('I-') else ('I-'+tag[2:] if tag.startswith('B-') else tag)])
            previous=wid
        aligned.append(ids)
    out['labels']=aligned
    return out

tokenized = ds.map(tokenize_and_align, batched=True, remove_columns=ds['train'].column_names)

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tokenizer)

import evaluate
seqeval = evaluate.load('seqeval')
def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    predictions = logits.argmax(-1)
    true_preds=[]; true_labels=[]
    for pred, lab in zip(predictions, label_ids):
        p=[]; l=[]
        for pi, li in zip(pred, lab):
            if li != -100: p.append(id2label[int(pi)]); l.append(id2label[int(li)])
        true_preds.append(p); true_labels.append(l)
    r=seqeval.compute(predictions=true_preds, references=true_labels)
    return {'precision':r['overall_precision'], 'recall':r['overall_recall'], 'f1':r['overall_f1'], 'accuracy':r['overall_accuracy']}

args = TrainingArguments(output_dir='./address-ner', learning_rate=3e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8, num_train_epochs=3, weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1', report_to='none', fp16=True)
trainer=Trainer(model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['test'], tokenizer=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
trainer.train()
trainer.evaluate()

In [ ]:
# Log in with a Hugging Face write token created at https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()
HF_REPO='YOUR_HF_USERNAME/indian-address-ner'
trainer.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print('Uploaded:', HF_REPO)

## Notes
- Synthetic data is a bootstrap dataset, not a substitute for human-labeled addresses.
- Before production, add real, consented/anonymized Indian addresses and measure performance on a held-out real test set.
- If Colab runs out of memory, lower batch size to 4 and/or max_length to 96.
